# Day 09. Exercise 00
# Regularization

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import joblib


## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df=pd.read_csv('../data/dayofweek.csv')
X=df.drop('dayofweek', axis=1)
y=df['dayofweek']
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)


## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [3]:
%%time
model = LogisticRegression(random_state=21,fit_intercept=False,solver='saga')

skf = StratifiedKFold(n_splits=10, random_state=21)

valid_scores = []

for train_idx, valid_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    
    model.fit(X_tr, y_tr)
    
    train_pred = model.predict(X_tr)
    valid_pred = model.predict(X_val)
    
    train_acc = accuracy_score(y_tr, train_pred)
    valid_acc = accuracy_score(y_val, valid_pred)
    
    valid_scores.append(valid_acc)
    
    print(f"train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}")

print(f"Average accuracy on crossval is {np.mean(valid_scores):.5f}")
print(f"Std is {np.std(valid_scores):.5f}")

c:\Users\Sofiya\AppData\Local\Programs\Python\Python38\lib\site-packages\sklearn\model_selection\_split.py:293: FutureWarning: Setting a random_state has no effect since shuffle is False. This will raise an error in 0.24. You should leave random_state to its default (None), or set shuffle=True.
  warnings.warn(


train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64221   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
CPU times: total: 1.08 s
Wall time: 1.56 s


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [4]:
%%time
opt_logreg_l1 = LogisticRegression(penalty='l1', solver='liblinear',C=1.0,random_state=21,fit_intercept=False)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
valid_scores = []

for train_idx, valid_idx in cv.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    
    opt_logreg_l1.fit(X_tr, y_tr)
    
    train_pred = opt_logreg_l1.predict(X_tr)
    valid_pred = opt_logreg_l1.predict(X_val)
    
    train_acc = accuracy_score(y_tr, train_pred)
    valid_acc = accuracy_score(y_val, valid_pred)
    
    valid_scores.append(valid_acc)

    print(f"train - {train_acc:.5f}   |   valid - {valid_acc:.5f}")

avg_valid_acc = np.mean(valid_scores)
std_valid_acc = np.std(valid_scores)

print(f"\nAverage accuracy on crossval is {avg_valid_acc:.5f}")
print(f"Std is {std_valid_acc:.5f}")

train - 0.62242   |   valid - 0.64444
train - 0.62242   |   valid - 0.60000
train - 0.62242   |   valid - 0.60000
train - 0.62737   |   valid - 0.62963
train - 0.63397   |   valid - 0.60000
train - 0.61913   |   valid - 0.59259
train - 0.63974   |   valid - 0.59259
train - 0.62737   |   valid - 0.52593
train - 0.61367   |   valid - 0.66418
train - 0.62768   |   valid - 0.57463

Average accuracy on crossval is 0.60240
Std is 0.03627
CPU times: total: 188 ms
Wall time: 552 ms


In [5]:
%%time
opt_logreg_l2 = LogisticRegression(penalty='l2',solver='lbfgs',C=1.0,random_state=21,fit_intercept=False)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []

for train_idx, valid_idx in cv.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    
    opt_logreg_l2.fit(X_tr, y_tr)
    
    train_pred = opt_logreg_l2.predict(X_tr)
    valid_pred = opt_logreg_l2.predict(X_val)
    
    train_acc = accuracy_score(y_tr, train_pred)
    valid_acc = accuracy_score(y_val, valid_pred)
    
    valid_scores.append(valid_acc)
    
    print(f"train - {train_acc:.5f}   |   valid - {valid_acc:.5f}")

avg_valid_acc = np.mean(valid_scores)
std_valid_acc = np.std(valid_scores)

print(f"\nAverage accuracy on crossval is {avg_valid_acc:.5f}")
print(f"Std is {std_valid_acc:.5f}")

train - 0.63974   |   valid - 0.65926
train - 0.63561   |   valid - 0.62222
train - 0.64386   |   valid - 0.60000
train - 0.64056   |   valid - 0.64444
train - 0.65375   |   valid - 0.60741
train - 0.62819   |   valid - 0.60000
train - 0.66035   |   valid - 0.60000
train - 0.63726   |   valid - 0.54074
train - 0.63756   |   valid - 0.66418
train - 0.64745   |   valid - 0.61194

Average accuracy on crossval is 0.61502
Std is 0.03399
CPU times: total: 11.8 s
Wall time: 2min 2s


In [6]:
%%time
opt_logreg_none = LogisticRegression(penalty='none', solver='lbfgs', random_state=21, fit_intercept=False,max_iter=1000)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
train_scores = []
valid_scores = []

for train_idx, valid_idx in cv.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    
    opt_logreg_none.fit(X_tr, y_tr)
    
    train_pred = opt_logreg_none.predict(X_tr)
    valid_pred = opt_logreg_none.predict(X_val)
    
    train_acc = accuracy_score(y_tr, train_pred)
    valid_acc = accuracy_score(y_val, valid_pred)
    
    valid_scores.append(valid_acc)
    
    print(f"train - {train_acc:.5f}   |   valid - {valid_acc:.5f}")

avg_valid_acc = np.mean(valid_scores)
std_valid_acc = np.std(valid_scores)

print(f"\nAverage accuracy on crossval is {avg_valid_acc:.5f}")
print(f"Std is {std_valid_acc:.5f}")

train - 0.65952   |   valid - 0.67407
train - 0.67189   |   valid - 0.60741
train - 0.65787   |   valid - 0.62963
train - 0.66117   |   valid - 0.66667
train - 0.66777   |   valid - 0.60741
train - 0.64633   |   valid - 0.62963
train - 0.66859   |   valid - 0.59259
train - 0.66447   |   valid - 0.60741
train - 0.65980   |   valid - 0.70149
train - 0.66969   |   valid - 0.62687

Average accuracy on crossval is 0.63432
Std is 0.03340
CPU times: total: 1min 15s
Wall time: 24.1 s


## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [7]:
%%time
baseline_svm = SVC(probability=True,kernel='linear',random_state=21)

cv = StratifiedKFold(n_splits=10, random_state=21)

train_scores = []
valid_scores = []

for train_idx, valid_idx in cv.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    
    baseline_svm.fit(X_tr, y_tr)
    
    train_pred = baseline_svm.predict(X_tr)
    valid_pred = baseline_svm.predict(X_val)
    
    train_acc = accuracy_score(y_tr, train_pred)
    valid_acc = accuracy_score(y_val, valid_pred)
    
    valid_scores.append(valid_acc)
    
    print(f"train - {train_acc:.5f}   |   valid - {valid_acc:.5f}")

avg_valid_acc = np.mean(valid_scores)
std_valid_acc = np.std(valid_scores)

print(f"\nAverage accuracy on crossval is {avg_valid_acc:.5f}")
print(f"Std is {std_valid_acc:.5f}")

c:\Users\Sofiya\AppData\Local\Programs\Python\Python38\lib\site-packages\sklearn\model_selection\_split.py:293: FutureWarning: Setting a random_state has no effect since shuffle is False. This will raise an error in 0.24. You should leave random_state to its default (None), or set shuffle=True.
  warnings.warn(


train - 0.70486   |   valid - 0.65926
train - 0.69662   |   valid - 0.75556
train - 0.69415   |   valid - 0.62222
train - 0.70239   |   valid - 0.65185
train - 0.69085   |   valid - 0.65185
train - 0.68920   |   valid - 0.64444
train - 0.69250   |   valid - 0.72593
train - 0.70074   |   valid - 0.62222
train - 0.69605   |   valid - 0.61940
train - 0.71087   |   valid - 0.63433

Average accuracy on crossval is 0.65871
Std is 0.04359
CPU times: total: 2.22 s
Wall time: 2.24 s


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [8]:
C_values = [0.01, 0.1, 1, 10, 100]

for C in C_values:
    
    model = SVC(C=C, kernel='linear', random_state=21)
    
    train_scores = []
    valid_scores = []
    
    for train_idx, valid_idx in cv.split(X_train, y_train):
        X_train_fold = X_train.iloc[train_idx]
        y_train_fold = y_train.iloc[train_idx]
        X_valid_fold = X_train.iloc[valid_idx]
        y_valid_fold = y_train.iloc[valid_idx]
        
        model.fit(X_train_fold, y_train_fold)
        
        y_pred_valid = model.predict(X_valid_fold)
        valid_acc = accuracy_score(y_valid_fold, y_pred_valid)
        valid_scores.append(valid_acc)
    
    avg_valid_acc = np.mean(valid_scores)
    std_valid_acc = np.std(valid_scores)
    
    print(f"With {C} average accuracy: {avg_valid_acc:.5f}")
    print(f"Std: {std_valid_acc:.5f}\n")

With 0.01 average accuracy: 0.37534
Std: 0.01848

With 0.1 average accuracy: 0.56230
Std: 0.02177

With 1 average accuracy: 0.65871
Std: 0.04359

With 10 average accuracy: 0.72771
Std: 0.04417

With 100 average accuracy: 0.75589
Std: 0.03550



## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [9]:
baseline_tree = DecisionTreeClassifier(max_depth=10,random_state=21)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)

train_scores = []
valid_scores = []

for train_idx, valid_idx in cv.split(X_train, y_train):
    X_train_fold = X_train.iloc[train_idx]
    y_train_fold = y_train.iloc[train_idx]
    X_valid_fold = X_train.iloc[valid_idx]
    y_valid_fold = y_train.iloc[valid_idx]
    
    baseline_tree.fit(X_train_fold, y_train_fold)
    
    y_pred_train = baseline_tree.predict(X_train_fold)
    y_pred_valid = baseline_tree.predict(X_valid_fold)
    
    train_acc = accuracy_score(y_train_fold, y_pred_train)
    valid_acc = accuracy_score(y_valid_fold, y_pred_valid)
    
    valid_scores.append(valid_acc)
    
    print(f"train - {train_acc:.5f}   |   valid - {valid_acc:.5f}")

avg_valid_acc = np.mean(valid_scores)
std_valid_acc = np.std(valid_scores)

print(f"\nAverage accuracy on crossval is {avg_valid_acc:.5f}")
print(f"Std is {std_valid_acc:.5f}")

train - 0.80874   |   valid - 0.77037
train - 0.79802   |   valid - 0.70370
train - 0.81286   |   valid - 0.73333
train - 0.80049   |   valid - 0.74815
train - 0.80956   |   valid - 0.70370
train - 0.78978   |   valid - 0.75556
train - 0.80709   |   valid - 0.62963
train - 0.82688   |   valid - 0.71111
train - 0.78995   |   valid - 0.79851
train - 0.80313   |   valid - 0.70896

Average accuracy on crossval is 0.72630
Std is 0.04409


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [10]:
max_depth = (2, 5, 10, 15, 20, 50, 100)

for depth in max_depth:
    print(f'max_depth = {depth}')
    valid_scores = []
    
    dt = DecisionTreeClassifier(max_depth=depth, random_state=21)

    for train_index, valid_index in cv.split(X_train, y_train):
        X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
        y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]

        dt.fit(X_train_fold, y_train_fold)

        train_pred = dt.predict(X_train_fold)
        valid_pred = dt.predict(X_valid_fold)

        train_score = accuracy_score(y_train_fold, train_pred)
        valid_score = accuracy_score(y_valid_fold, valid_pred)

        valid_scores.append(valid_score)

        print(f'train -  {train_score:.5f}   |   valid -  {valid_score:.5f}')


    average_accuracy = np.mean(valid_scores)
    std_accuracy = np.std(valid_scores)

    print(f'Average accuracy on crossval is {average_accuracy:.5}')
    print(f'Std is {std_accuracy:.5}')
    print()

max_depth = 2
train -  0.43199   |   valid -  0.45926
train -  0.43116   |   valid -  0.46667
train -  0.43281   |   valid -  0.45185
train -  0.43281   |   valid -  0.45185
train -  0.43611   |   valid -  0.42222
train -  0.43776   |   valid -  0.40741
train -  0.43446   |   valid -  0.43704
train -  0.43364   |   valid -  0.44444
train -  0.43740   |   valid -  0.41045
train -  0.43904   |   valid -  0.39552
Average accuracy on crossval is 0.43467
Std is 0.023103

max_depth = 5
train -  0.58285   |   valid -  0.61481
train -  0.57626   |   valid -  0.52593
train -  0.61253   |   valid -  0.60000
train -  0.58862   |   valid -  0.58519
train -  0.58615   |   valid -  0.51111
train -  0.56224   |   valid -  0.53333
train -  0.58120   |   valid -  0.51852
train -  0.62407   |   valid -  0.51111
train -  0.57414   |   valid -  0.56716
train -  0.56672   |   valid -  0.48507
Average accuracy on crossval is 0.54522
Std is 0.041345

max_depth = 10
train -  0.80874   |   valid -  0.77037
tra

In [11]:
param_grid = {
    'max_depth': [5, 7, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(DecisionTreeClassifier(random_state=21),param_grid,cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=21),scoring='accuracy',n_jobs=-1)

grid_search.fit(X, y)

print("Лучшие параметры:", grid_search.best_params_)
print("Лучшая accuracy:", grid_search.best_score_)

Лучшие параметры: {'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
Лучшая accuracy: 0.8973978543711482


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [12]:
rf = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=21)
valid_scores = []

for train_index, valid_index in cv.split(X_train, y_train):
    X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
    
    rf.fit(X_train_fold, y_train_fold)
    
    train_pred = rf.predict(X_train_fold)
    valid_pred = rf.predict(X_valid_fold)
    
    train_score = accuracy_score(y_train_fold, train_pred)
    valid_score = accuracy_score(y_valid_fold, valid_pred)
    
    valid_scores.append(valid_score)
    
    print(f'train -  {train_score:.5f}   |   valid -  {valid_score:.5f}')


average_accuracy = np.mean(valid_scores)
std_accuracy = np.std(valid_scores)

print(f'Average accuracy on crossval is {average_accuracy:.5}')
print(f'Std is {std_accuracy:.5}')

train -  0.97362   |   valid -  0.86667
train -  0.96867   |   valid -  0.83704
train -  0.96455   |   valid -  0.89630
train -  0.97197   |   valid -  0.92593
train -  0.97115   |   valid -  0.88889
train -  0.95548   |   valid -  0.89630
train -  0.96373   |   valid -  0.88148
train -  0.97362   |   valid -  0.88148
train -  0.97117   |   valid -  0.92537
train -  0.96458   |   valid -  0.85821
Average accuracy on crossval is 0.88577
Std is 0.02636


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [13]:
for max_depth in (10, 50, 100):
    for n_estimators in (100, 200, 300):
        
        print(f'max_depth = {max_depth}')
        print(f'n_estimators = {n_estimators}')
        
        rf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=21)

        valid_scores = []

        for train_index, valid_index in cv.split(X_train, y_train):
            X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
            y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]

            rf.fit(X_train_fold, y_train_fold)

            train_pred = rf.predict(X_train_fold)
            valid_pred = rf.predict(X_valid_fold)

            train_score = accuracy_score(y_train_fold, train_pred)
            valid_score = accuracy_score(y_valid_fold, valid_pred)

            valid_scores.append(valid_score)

            print(f'train -  {train_score:.5f}   |   valid -  {valid_score:.5f}')


        average_accuracy = np.mean(valid_scores)
        std_accuracy = np.std(valid_scores)

        print(f'Average accuracy on crossval is {average_accuracy:.5}')
        print(f'Std is {std_accuracy:.5}')
        print()

max_depth = 10
n_estimators = 100
train -  0.86974   |   valid -  0.80741
train -  0.86645   |   valid -  0.77037
train -  0.89860   |   valid -  0.82222
train -  0.88046   |   valid -  0.81481
train -  0.86480   |   valid -  0.77778
train -  0.88953   |   valid -  0.82222
train -  0.89118   |   valid -  0.79259
train -  0.87881   |   valid -  0.78519
train -  0.88880   |   valid -  0.88806
train -  0.87150   |   valid -  0.72388
Average accuracy on crossval is 0.80045
Std is 0.040564

max_depth = 10
n_estimators = 200
train -  0.88458   |   valid -  0.81481
train -  0.86810   |   valid -  0.77778
train -  0.89448   |   valid -  0.81481
train -  0.90354   |   valid -  0.82963
train -  0.87387   |   valid -  0.78519
train -  0.87139   |   valid -  0.80000
train -  0.88623   |   valid -  0.77778
train -  0.88788   |   valid -  0.77778
train -  0.88386   |   valid -  0.87313
train -  0.88633   |   valid -  0.73881
Average accuracy on crossval is 0.79897
Std is 0.034786

max_depth = 10
n_e

In [14]:
for min_samples_leaf in (2, 5, 10):

    print(f'max_depth = 100')
    print(f'n_estimators = 300')
    print(f'min_samples_leaf = {min_samples_leaf}')

    rf = RandomForestClassifier(n_estimators=300, max_depth=100, random_state=21)

    valid_scores = []

    for train_index, valid_index in cv.split(X_train, y_train):
        X_train_fold, X_valid_fold = X_train.iloc[train_index], X_train.iloc[valid_index]
        y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]

        rf.fit(X_train_fold, y_train_fold)

        train_pred = rf.predict(X_train_fold)
        valid_pred = rf.predict(X_valid_fold)

        train_score = accuracy_score(y_train_fold, train_pred)
        valid_score = accuracy_score(y_valid_fold, valid_pred)

        valid_scores.append(valid_score)

        print(f'train -  {train_score:.5f}   |   valid -  {valid_score:.5f}')


    average_accuracy = np.mean(valid_scores)
    std_accuracy = np.std(valid_scores)

    print(f'Average accuracy on crossval is {average_accuracy:.5}')
    print(f'Std is {std_accuracy:.5}')
    print()

max_depth = 100
n_estimators = 300
min_samples_leaf = 2
train -  1.00000   |   valid -  0.88889
train -  1.00000   |   valid -  0.87407
train -  1.00000   |   valid -  0.94815
train -  1.00000   |   valid -  0.91852
train -  1.00000   |   valid -  0.92593
train -  1.00000   |   valid -  0.91852
train -  1.00000   |   valid -  0.94815
train -  1.00000   |   valid -  0.88889
train -  1.00000   |   valid -  0.94776
train -  1.00000   |   valid -  0.89552
Average accuracy on crossval is 0.91544
Std is 0.026136

max_depth = 100
n_estimators = 300
min_samples_leaf = 5
train -  1.00000   |   valid -  0.88889
train -  1.00000   |   valid -  0.87407
train -  1.00000   |   valid -  0.94815
train -  1.00000   |   valid -  0.91852
train -  1.00000   |   valid -  0.92593
train -  1.00000   |   valid -  0.91852
train -  1.00000   |   valid -  0.94815
train -  1.00000   |   valid -  0.88889
train -  1.00000   |   valid -  0.94776
train -  1.00000   |   valid -  0.89552
Average accuracy on crossval is

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [15]:
rf = RandomForestClassifier(max_depth=50, n_estimators = 100, random_state=21)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
accuracy_score(y_test, pred)

0.9378698224852071

In [16]:
df_pred = pd.DataFrame()
df_pred['prediction'] = pred
df_pred['y_test'] = np.array(y_test)
df_pred['error'] = df_pred['prediction'] != df_pred['y_test']
df_pred.groupby('y_test').mean()['error'].sort_values(ascending=False) * 100

y_test
0    25.925926
2    10.000000
4     9.523810
5     5.555556
1     5.454545
3     2.500000
6     1.408451
Name: error, dtype: float64

In [17]:
joblib.dump(rf, 'best_random_forest_model.joblib')

['best_random_forest_model.joblib']